In [2]:
!pip install pymupdf


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 69.7 MB/s eta 0:00:00:00:0100:01


In [3]:
import json
import fitz  # PyMuPDF
import re
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
#from transformers import BitsAndBytesConfig

In [4]:
pdf_path="/kaggle/input/datasets/koushikikundu/hr-policies/HR Policy Manual 2023 (8).pdf"

doc = fitz.open(pdf_path)

In [5]:
pages = []

for i, page in enumerate(doc):
    # Find tables on the current page
    tabs = page.find_tables()
    
    if tabs.tables:
        # Extract table data structured as lists/dataframes
        extracted_tables = [tab.extract() for tab in tabs]
        
        # Get bounding boxes of all detected tables
        table_bboxes = [tab.bbox for tab in tabs]
        
        # Extract page text while ignoring text inside table bounding boxes
        text_page = page.get_text("words")  # list of (x0, y0, x1, y1, word, block_no, line_no, word_no)
        non_table_words = []
        
        for word_info in text_page:
            word_bbox = fitz.Rect(word_info[:4])
            # Check if word falls inside any table bbox
            in_table = any(word_bbox.intersects(tbl_box) for tbl_box in table_bboxes)
            if not in_table:
                non_table_words.append(word_info[4])
        
        page_text = " ".join(non_table_words)
        
        pages.append({
            "page": i + 1,
            "text": page_text.strip(),
            "tables": extracted_tables
        })
    else:
        pages.append({
            "page": i + 1,
            "text": page.get_text("text").strip()
        })

print("Total pages:", len(pages))

Consider using the pymupdf_layout package for a greatly improved page layout analysis.
Total pages: 208


In [6]:
pages=pages[10:37]+pages[39:41]+pages[54:86]+pages[94:115]+pages[116:123]+pages[124:138]+pages[152:164]+pages[165:176]+pages[177:194]

In [7]:
def normalize(s):
    s = s.replace("\xa0", " ")
    s = re.sub(r"Page\s+\d+", " ", s)
    s=s.replace("IIMA HR Policy Manual 2023", "")
    s = re.sub(r"\n+", "\n",s)
    s= re.sub(r" +", " ", s)
    return s.strip()
    

for page in pages:
    text = normalize(page["text"])   

    if "tables" in page:

        cleaned_tables = []
        for table in page["tables"]:

            cleaned_table = []

            for row in table:

                cleaned_row = []
                for cell in row:

                    cell = "" if cell is None else cell.strip()

                    # Clean the cell
                    cell = normalize(cell)

                    cleaned_row.append(cell)

                cleaned_table.append(cleaned_row)

            cleaned_tables.append(cleaned_table)

        page["tables"] = cleaned_tables

    page["text"] = text

In [8]:
def tables_to_text(tables):
    if not tables:
        return ""

    output = []

    for idx, table in enumerate(tables, start=1):
        output.append(f"\nTable {idx}")

        for row in table:
            row = [
                str(cell).replace("\n", " ").strip()
                for cell in row
                if cell is not None and str(cell).strip() != ""
            ]

            if row:
                output.append(" | ".join(row))

    return "\n".join(output)

In [9]:
page_contents = []

for page in pages:

    page_text = page["text"].strip()
    if 'tables' in page:

        table_text = tables_to_text(page["tables"])

        combined = page_text + "\n\n" + table_text
    else:
        combined = page_text

    page_contents.append(combined)

In [10]:
page_contents[5]

'6\n\n\nTable 1\n8. | Prof. Ajay Pandey IIM Ahmedabad | Chairman’s Nominee\n9. | Prof. Sachin Jayaswal IIM Ahmedabad | Chairman’s Nominee\n10. | Ramesh Mangaleswaran Senior Partner Emeritus, McKinsey & Company Chennai, Tamil Nadu, India | Co-opted by the Board from the Alumni\n11. | Dr. Hasit Joshipura Advisor to L&T Group CEO and MD, Data Centre & Cloud, Innovation Fund Larsen & Toubro Limited Landmark A Wing, 5th Floor, Suren Road Off. Andheri-Kurla Road Andheri (East), Mumbai – 400093 | -do-\n12. | Rama Bijapurkar 206, Nirman Kendra, Dr. E. Moses Road, Mahalakshmi, Mumbai 400 011. | -do-\n13. | Prof. Pradeep K. Chintagunta Joseph T. and Bernice S. Lewis Distinguished Service Professor of Marketing University of Chicago Booth School of Business Chicago, IL 60637, USA | -do-\n14. | Samir U. Mehta Chairman, Torrent Group Torrent House, Off. Ashram Road, Ahmedabad – 380009 | Co-opted by the Board from the members of Society\n15. | Prof. Errol D’Souza Director IIM Ahmedabad | Ex-Officio\

In [11]:
def make_chunks(page_contents,
                pages_per_chunk=3,
                overlap=1):

    chunks = []

    step = pages_per_chunk - overlap

    for start in range(0, len(page_contents), step):

        chunk = "\n\n".join(
            page_contents[start:start+pages_per_chunk]
        )

        chunks.append(chunk)

    return chunks

In [12]:
chunks = make_chunks(
    page_contents,
    pages_per_chunk=3,
    overlap=1
)

In [13]:
len(chunks)

72

In [14]:
model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16
).cuda()

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [15]:
def create_prompt(chunk):

    return f"""
You are an HR policy expert.

Generate 5 high-quality Question-Answer pairs.

Instructions:
- Use only the information given.
- Use information from both the text and tables.
- Do not make up information.
- Questions should sound like employees asking HR.
- Answers should be complete and precise.
- Return ONLY valid JSON.

Format:

[
    {{
        "question":"...",
        "answer":"..."
    }}
]

Policy:

{chunk}
"""

In [ ]:
dataset = []
for chunk in chunks:

    prompt = create_prompt(chunk)

    messages = [
        {
            "role":"system",
            "content":"You create datasets for HR chatbots."
        },
        {
            "role":"user",
            "content":prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=2000,
        temperature=0.2,
        do_sample=False
    )

    response = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    )

    try:
        qa = json.loads(response)
        with open("hr_policy_qa.jsonl", "a", encoding="utf-8") as f:
            f.write(json.dumps(qa, ensure_ascii=False) + "\n")
            print("Inserted")
        dataset.extend(qa)
    except:
        print("Failed to parse one chunk")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [ ]:
dataset